In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/diabetes-dataset/train_dev_test_data.pkl


## In this notebook, the same model for diabetes prediction will be implmented using PyTorch and introducing Sparsity to the Neural Network, notably L1 Regularization on activations.

In [7]:
### Sparsity auto encoder style 
import pickle
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim
import torch.nn.functional as F

############################################
# 1. Load previously split dataset
############################################

with open("/kaggle/input/diabetes-dataset/train_dev_test_data.pkl", "rb") as f:
    X_train, X_dev, X_test, y_train, y_dev, y_test = pickle.load(f)

############################################
# 2. Convert data to tensors
############################################

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_dev_tensor = torch.tensor(X_dev.values, dtype=torch.float32)
y_dev_tensor = torch.tensor(y_dev.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

############################################
# 3. Create datasets and dataloaders
############################################

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
dev_dataset = TensorDataset(X_dev_tensor, y_dev_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

############################################
# 4. Sparse Classifier Model
############################################

class SparseClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 6)
        self.fc2 = nn.Linear(6, 6)
        self.fc3 = nn.Linear(6, 5)
        self.out = nn.Linear(5, 1)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))
        h3 = F.relu(self.fc3(h2))
        logits = self.out(h3)

        # Return logits + hidden activations
        return logits, [h1, h2, h3]

model = SparseClassifier(X_train_tensor.shape[1])
print(model)

############################################
# 5. Loss, Optimizer, Sparsity Settings
############################################

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

lambda_sparse = 1e-3  # sparsity strength
num_epochs = 50

def sparsity_loss(hidden_activations):
    # L1 penalty on activations (autoencoder-style)
    return sum(h.abs().mean() for h in hidden_activations)

############################################
# 6. Training Loop
############################################

for epoch in range(num_epochs):
    model.train()
    train_loss_epoch = 0
    correct_train = 0
    total_train = 0

    for X_batch, y_batch in train_loader:
        y_batch = y_batch.view(-1, 1).float()

        logits, hidden = model(X_batch)

        task_loss = criterion(logits, y_batch)
        sparse_penalty = sparsity_loss(hidden)
        loss = task_loss + lambda_sparse * sparse_penalty

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_epoch += loss.item() * X_batch.size(0)

        preds = torch.sigmoid(logits) >= 0.5
        correct_train += (preds.float() == y_batch).sum().item()
        total_train += y_batch.size(0)

    train_loss_epoch /= len(train_loader.dataset)
    train_acc = correct_train / total_train

    ########################################
    # Dev Evaluation
    ########################################

    model.eval()
    with torch.no_grad():
        dev_logits, _ = model(X_dev_tensor)
        dev_loss = criterion(dev_logits, y_dev_tensor.view(-1, 1).float())

        preds_dev = torch.sigmoid(dev_logits) >= 0.5
        correct_dev = (preds_dev.float() == y_dev_tensor.view(-1, 1).float()).sum().item()
        dev_acc = correct_dev / len(y_dev_tensor)

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss_epoch:.4f}, Train Acc: {train_acc:.4f} | "
        f"Dev Loss: {dev_loss.item():.4f}, Dev Acc: {dev_acc:.4f}"
    )

############################################
# 7. Test Evaluation
############################################

model.eval()
with torch.no_grad():
    test_logits, _ = model(X_test_tensor)
    preds_test = torch.sigmoid(test_logits) >= 0.5
    test_acc = (preds_test.float() == y_test_tensor.view(-1, 1).float()).sum().item() / len(y_test_tensor)

print(f"\n✅ Test Accuracy: {test_acc:.4f}")


SparseClassifier(
  (fc1): Linear(in_features=11, out_features=6, bias=True)
  (fc2): Linear(in_features=6, out_features=6, bias=True)
  (fc3): Linear(in_features=6, out_features=5, bias=True)
  (out): Linear(in_features=5, out_features=1, bias=True)
)
Epoch 1/50 | Train Loss: 0.6317, Train Acc: 0.6645 | Dev Loss: 0.4817, Dev Acc: 0.9127
Epoch 2/50 | Train Loss: 0.3720, Train Acc: 0.9159 | Dev Loss: 0.3170, Dev Acc: 0.9127
Epoch 3/50 | Train Loss: 0.2975, Train Acc: 0.9159 | Dev Loss: 0.2903, Dev Acc: 0.9127
Epoch 4/50 | Train Loss: 0.2691, Train Acc: 0.9159 | Dev Loss: 0.2585, Dev Acc: 0.9127
Epoch 5/50 | Train Loss: 0.2337, Train Acc: 0.9164 | Dev Loss: 0.2218, Dev Acc: 0.9156
Epoch 6/50 | Train Loss: 0.2001, Train Acc: 0.9258 | Dev Loss: 0.1908, Dev Acc: 0.9288
Epoch 7/50 | Train Loss: 0.1739, Train Acc: 0.9374 | Dev Loss: 0.1679, Dev Acc: 0.9392
Epoch 8/50 | Train Loss: 0.1570, Train Acc: 0.9478 | Dev Loss: 0.1541, Dev Acc: 0.9469
Epoch 9/50 | Train Loss: 0.1478, Train Acc: 0.9540 

In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

############################################
# 8. Detailed Evaluation Metrics
############################################

model.eval()
with torch.no_grad():
    logits, _ = model(X_test_tensor)
    probs = torch.sigmoid(logits).cpu().numpy().ravel()
    y_true = y_test_tensor.cpu().numpy()

# Binary predictions
y_pred = (probs >= 0.5).astype(int)

# Metrics
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)      # Sensitivity
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, probs)

cm = confusion_matrix(y_true, y_pred)

print("\n📊 Test Set Evaluation Metrics")
print(f"Accuracy     : {acc:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1-score     : {f1:.4f}")
print(f"ROC-AUC      : {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred))


📊 Test Set Evaluation Metrics
Accuracy     : 0.9599
Precision    : 0.8920
Recall       : 0.6319
F1-score     : 0.7398
ROC-AUC      : 0.9641

Confusion Matrix:
[[9028   69]
 [ 332  570]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98      9097
         1.0       0.89      0.63      0.74       902

    accuracy                           0.96      9999
   macro avg       0.93      0.81      0.86      9999
weighted avg       0.96      0.96      0.96      9999

